# Valosaber batch processor

Turns replays (`.vrf`) into small checksummed **shards** for the Valosaber analyst, and keeps the raw export zipped as a backup.

**Before you run it**
1. Put your `.vrf` replays in Google Drive at `MyDrive/valosaber/replays/`.
2. The bootstrap zip (it holds `vrfkit`) must be in `MyDrive/valosaber/bin/`.
3. In Colab: the key icon (Secrets) -> add `GITHUB_TOKEN`, a fine-grained token limited to the `ValorantUtility` repo (Contents: read and write), and switch on *Notebook access*. Never paste the token into a cell.
4. Runtime -> Run all. It stops by itself if a check fails.

Only **Swiftplay** replays are verified. Standard-mode replays are processed too, but their shards carry a warning and are not pushed unless you allow it in the config cell.

In [ ]:
# ---- config ----
DRIVE_ROOT   = "/content/drive/MyDrive/valosaber"
REPLAYS_DIR  = f"{DRIVE_ROOT}/replays"
OUTBOX_DIR   = f"{DRIVE_ROOT}/outbox"
REPO_URL     = "https://github.com/nicholasjlivingston-a11y/ValorantUtility.git"
REPO_DIR     = "/content/ValorantUtility"
LIMIT        = 0        # 0 = every replay; set e.g. 2 for a first test
KEEP_EXPORT_ZIP = True  # raw export zip per match, saved to Drive (backup)
PUSH_STANDARD_SHARDS = False  # standard-mode shard logic is untested

import platform
assert platform.system() == "Linux" and platform.machine() in ("x86_64", "AMD64"), "vrfkit needs Linux x86-64"
from google.colab import drive
drive.mount("/content/drive")
import os; os.makedirs(OUTBOX_DIR, exist_ok=True)
print("replays:", sorted(os.listdir(REPLAYS_DIR))[:20] if os.path.isdir(REPLAYS_DIR) else "FOLDER MISSING: " + REPLAYS_DIR)

In [ ]:
# ---- get the code (private repo: token from Colab Secrets, never stored in git config) ----
import base64, subprocess, os
from google.colab import userdata
try:
    tok = userdata.get("GITHUB_TOKEN")
except Exception as e:
    raise SystemExit("Add GITHUB_TOKEN under Secrets (key icon) and enable Notebook access, then rerun. " + str(e))
HDR = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{tok}".encode()).decode()
def git(*args, cwd=None):
    r = subprocess.run(["git", "-c", f"http.extraheader={HDR}", *args], cwd=cwd, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip()[-600:])
    return r.returncode
if os.path.isdir(REPO_DIR + "/.git"):
    assert git("pull", "--rebase", "origin", "main", cwd=REPO_DIR) == 0
else:
    assert git("clone", REPO_URL, REPO_DIR) == 0
os.chdir(REPO_DIR)
print("repo ready at", os.getcwd())

In [ ]:
# ---- verify vrfkit (hash must match parser/vrfkit.sha256; refuses otherwise; never runs it) ----
!python3 parser/colab_bootstrap.py
!ls -l bin/vrfkit

In [ ]:
# ---- process every replay (safe to rerun: finished matches are skipped, failed ones retried) ----
limit = f"--limit {LIMIT}" if LIMIT else ""
nozip = "" if KEEP_EXPORT_ZIP else "--no-zip"
!python3 parser/batch_process.py --replays "{REPLAYS_DIR}" --outbox "{OUTBOX_DIR}" --vrfkit bin/vrfkit --lookup data/name_lookup_build13_08.json {limit} {nozip}

In [ ]:
# ---- report ----
import pandas as pd, glob, json
rep = f"{OUTBOX_DIR}/batch_report.csv"
if os.path.exists(rep):
    display(pd.read_csv(rep))
for sp in sorted(glob.glob(f"{OUTBOX_DIR}/*/status.json")):
    s = json.load(open(sp))
    if s["overall"] != "ok":
        bad = [k for k, v in s["steps"].items() if not v.get("ok")]
        print(s["match_id"], s["overall"], bad, s.get("exception", "")[-300:], *[s["steps"][k].get("error", "")[-300:] for k in bad])

In [ ]:
# ---- send finished shards to the repo inbox (small files only; the raw export zips stay in Drive) ----
import shutil, glob, json
inbox = f"{REPO_DIR}/inbox"
added = []
for sp in sorted(glob.glob(f"{OUTBOX_DIR}/*/status.json")):
    s = json.load(open(sp)); mid = s["match_id"]
    shard = f"{OUTBOX_DIR}/{mid}/shard"
    if s["overall"] != "ok" or not os.path.isdir(shard): continue
    if s.get("warning") and not PUSH_STANDARD_SHARDS:
        print("held back (untested mode):", mid); continue
    dest = f"{inbox}/shard-{mid}"
    if os.path.exists(dest): continue
    shutil.copytree(shard, dest); added.append(mid)
print("new shards:", added)
if added:
    git("config", "user.name", "valosaber-colab", cwd=REPO_DIR)
    git("config", "user.email", "valosaber-colab@users.noreply.github.com", cwd=REPO_DIR)
    git("add", "inbox", cwd=REPO_DIR)
    git("commit", "-m", f"Add {len(added)} shard(s) from Colab batch", cwd=REPO_DIR)
    git("pull", "--rebase", "origin", "main", cwd=REPO_DIR)
    git("push", "origin", "main", cwd=REPO_DIR)

## Then
Tell Claude **"process the inbox"**. Claude pulls the repo, verifies each shard (schema, SHA-256, row counts), merges it into the warehouse, and updates the vault.

If a match shows `export_only` or `failed`, send Claude the line printed above, or the `status.json` in `outbox/<match>/`. Don't retry blindly: the status says which step failed.